# A/B testing d’une campagne CRM Olist

L’historique Olist fournit un taux de référence de 290 rachats sur 39 964 clients. Les résultats de campagne ci-dessous sont simulés. On assigne les clients au hasard, pour moitié au groupe campagne et pour moitié au groupe témoin. La métrique principale est le rachat à 90 jours par client assigné.

In [1]:
from mktscore.ab_stats import p_valeur_srm, ratio_et_erreur, taille_par_groupe

taux_reference = 290 / 39_964
taux_attendu = taux_reference + 0.002
n_par_groupe = taille_par_groupe(taux_reference, taux_attendu)
print(f"Taux de référence : {taux_reference:.2%}")
print(f"Effet minimal : +0,20 point, soit {0.002 / taux_reference:.1%} en relatif")
print(f"Clients nécessaires par groupe : {n_par_groupe:,}")
print(f"Clients nécessaires au total : {2 * n_par_groupe:,}")

Taux de référence : 0.73%
Effet minimal : +0,20 point, soit 27.6% en relatif
Clients nécessaires par groupe : 32,134
Clients nécessaires au total : 64,268


## Vérifier l’assignation

Le test de déséquilibre d’échantillon (SRM) compare les effectifs observés aux proportions prévues. On l’applique aux clients assignés, avant tout filtre sur la réponse ou l’achat. Dans cet exemple, une p-valeur inférieure à 0,001 déclenche une investigation avant l’interprétation de l’effet.

In [2]:
n_controle = n_par_groupe
n_campagne = n_par_groupe
print(f"SRM, groupes équilibrés : p = {p_valeur_srm(n_controle, n_campagne):.3f}")
print(f"SRM, incident simulé : p = {p_valeur_srm(10_600, 9_400):.3g}")

SRM, groupes équilibrés : p = 1.000
SRM, incident simulé : p = 2.15e-17


## Estimer l’effet de la campagne

Pour le taux de rachat, chaque client apporte une valeur de zéro ou un au numérateur et un au dénominateur. La méthode Delta donne l’erreur standard d’un ratio de totaux à partir de ces paires par client. Pour comparer deux groupes indépendants, on additionne leurs variances. L’intervalle à 95 % utilise l’approximation normale.

In [3]:
from math import hypot
from statistics import NormalDist

rachats_controle = round(n_controle * taux_reference)
rachats_campagne = round(n_campagne * taux_attendu)

def estimer_taux(rachats: int, clients: int) -> tuple[float, float]:
    resultats = [1.0] * rachats + [0.0] * (clients - rachats)
    return ratio_et_erreur(resultats, [1.0] * clients)

taux_controle, erreur_controle = estimer_taux(rachats_controle, n_controle)
taux_campagne, erreur_campagne = estimer_taux(rachats_campagne, n_campagne)
effet = taux_campagne - taux_controle
marge = NormalDist().inv_cdf(0.975) * hypot(erreur_controle, erreur_campagne)
print(f"Rachat témoin : {taux_controle:.2%}")
print(f"Rachat campagne : {taux_campagne:.2%}")
print(f"Effet : {100 * effet:+.2f} point, IC 95 % [{100 * (effet - marge):.2f} ; {100 * (effet + marge):.2f}]")

Rachat témoin : 0.73%
Rachat campagne : 0.92%
Effet : +0.20 point, IC 95 % [0.06 ; 0.34]


Dans un vrai test, l’écart entre les groupes estime les rachats supplémentaires attribuables à la campagne si l’assignation et la mesure ont été respectées. Un effet de +0,20 point correspond à environ deux rachats supplémentaires par 1 000 clients assignés à la campagne.